In [2]:
import os
import sys
sys.path.insert(
    0, os.path.abspath('../../')
)

import json
import yaml

from pathlib import Path
from rich.console import Console
from rich.table import Table

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [5]:
root_dir = Path("../../").resolve()
print("Root directory:", root_dir)

Root directory: /home/hgkahng/Workspaces/soft-prompt


In [6]:
N_train = 8_544

## 1. Load Synthetic Data

In [50]:
load_dir = root_dir / "results/sst/gemini-2.0-flash/soft+cot"
print("Model directory:", load_dir)
print(*os.listdir(load_dir), sep="\n")

Model directory: /home/hgkahng/Workspaces/soft-prompt/results/sst/gemini-2.0-flash/soft+cot
template.jsonl
embeddings
config.yaml
data.jsonl
template_formatted.txt


In [51]:
# Print configurations

with open(load_dir / 'config.yaml') as f:
    cfg = yaml.safe_load(f)

style = "red" if cfg['hard'] else "green"

table = Table(title="Configuration(s)")
table.add_column("Name", justify="right", style="white", no_wrap=True)
table.add_column("Value", justify="left", style=style)
_ = [table.add_row(k, str(v)) for k, v in cfg.items()]

console = Console()
console.print(table)

         Configuration(s)          
┏━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃         Name ┃ Value            ┃
┡━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│   batch_size │ 50               │
│          cot │ True             │
│         data │ sst2             │
│         hard │ False            │
│ log_interval │ 2                │
│   max_tokens │ 16384            │
│        model │ gemini-2.0-flash │
│ num_examples │ None             │
│  sample_size │ 20000            │
│  temperature │ 1.0              │
└──────────────┴──────────────────┘

In [52]:
with open(load_dir / "data.jsonl", "r") as f:
    data = [json.loads(line) for line in f]

print(f"Data size: {len(data):,}")

Data size: 20,049


In [53]:
data[-1]

{'label': 0.018,
 'text': 'Utterly boring and a complete waste of time. The acting was terrible, and the plot made no sense. Avoid at all costs!',
 'reasoning': 'The sentiment is extremely negative. The review should reflect strong disappointment and dislike. It should be short and to the point.'}

In [54]:
# labels
labels = [d['label'] for d in data]
labels = np.array(labels)

soft_labels = labels.copy()
hard_labels = (soft_labels > 0.5).astype(int)

print("Hard labels, shape:", hard_labels.shape)
print("Soft labels, shape:", soft_labels.shape)

Hard labels, shape: (20049,)
Soft labels, shape: (20049,)


In [55]:
# text
texts = [d['text'] for d in data]
len(texts)

20049

In [56]:
# embeddings
embeddings = np.load(
    load_dir / "embeddings/openai/text-embedding-3-small/data.npy"
)
print(embeddings.shape)

(20049, 1536)


In [57]:
assert len(texts) == embeddings.shape[0]
assert len(texts) == labels.shape[0]
print(len(texts))

20049


In [58]:
%%time
rng = np.random.default_rng(42)
sample_idx = rng.permutation(embeddings.shape[0])[:N_train]

labels = labels[sample_idx]
embeddings = embeddings[sample_idx]
texts = [t for i, t in enumerate(texts) if i in sample_idx]

assert len(texts) == embeddings.shape[0]
assert len(texts) == labels.shape[0]
print(len(texts))

8544
CPU times: user 57.2 ms, sys: 10 ms, total: 67.2 ms
Wall time: 66.8 ms


## 2. Diversity Metrics

In [59]:
from softprompt.metrics.diversity import (
    vocabulary_size,
    distinct_n,
    average_pairwise_similarity,
    average_pairwise_similarity_by_class,
    inter_sample_ngram_freq
)

In [60]:
vocab_size = vocabulary_size(texts)
print(f"Vocab: {vocab_size:>7,}")

Vocab:   1,604


In [61]:
distinct_2 = distinct_n(texts, n=2)
print(f"Distinct-2: {distinct_2:.4f}")

Distinct-2: 0.0447


In [62]:
%%time
inter_sim, intra_sim = average_pairwise_similarity_by_class(
    embeddings, labels
)
print(f"Inter-class APS: {inter_sim:.4f}\n",
      f"Intra-class APS: {intra_sim:.4f}")

Inter-class APS: 0.4841
 Intra-class APS: 0.6739
CPU times: user 7.13 s, sys: 1.7 s, total: 8.83 s
Wall time: 474 ms


In [63]:
%%time
avg_ps = average_pairwise_similarity(embeddings)
print(f"Avg Pairwise Similarity: {avg_ps:.4f}")

Avg Pairwise Similarity: 0.4843
CPU times: user 7.39 s, sys: 2.09 s, total: 9.47 s
Wall time: 568 ms
